[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/_energy_track/E4_Many_To_Many_Demand.ipynb)

# Many-to-Many on the Demand Series
--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

Two ways an RNN can put out more than one number:

- **(a) Two targets at once** - next-hour demand *and* next-hour temperature from one model (`Dense(2)`), borrowing strength across both.
- **(b) Multi-step ahead** - the next **24 hours** of demand from the past week (`Dense(24)`), with error plotted by horizon against the seasonal-naive forecast. This is the forecast the dispatch desk actually wants.

*Energy-track version of the two BDL many-to-many notebooks (dew point + pressure; temperature +1h).*

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 15B — Many-to-many: two targets at once, then 24 hours ahead
- (a) Two targets: put BOTH targets on the right, chop from the right with n_targets=2, Dense(2, linear). One model, two MAEs - in their own units after inverse_transform.
- (b) Multi-step: past week (168) -> next 24 hours, Dense(24). The tensor is (samples, 168, 8) -> (samples, 24).
- The plot that matters: MAE by horizon. Hour 1 is nearly free; hour 24 is the honest number. The fair baseline a day ahead is seasonal naive (~227), not persistence.
- Show one day-ahead forecast against reality - the model gets the shape, misses the peak height.
- Close on save/reload and the on-your-own: a holiday flag, GRU swap, test on 2020 (distribution shift).
-->


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, classification_report, confusion_matrix
from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout, SimpleRNN, LSTM, GRU, Bidirectional, Conv1D, MaxPooling1D, Flatten
from keras.callbacks import EarlyStopping
import keras
keras.utils.set_random_seed(5509)   # reproducibility: same numbers every run (CPU exact; a GPU may drift a little)

## Read, sort, split

In [2]:
url = "https://raw.githubusercontent.com/drdave-teaching/OPIM5509Files/main/OPIM5509_Module4_Files/data/BDL_cleanweather_energy.csv"
df = pd.read_csv(url, parse_dates=["Datetime"])
print("in date order as delivered?", df["Datetime"].is_monotonic_increasing)      # it is NOT - always check
df = df.sort_values("Datetime").set_index("Datetime").ffill()

# two years is plenty for the lecture: train on 2018, test on 2019 - chronological, never shuffled
data  = df.loc["2018-01-01":"2019-12-31", ["BDL_tmpf", "BDL_dwpf", "BDL_relh", "Demand"]].copy()
data["hour_sin"] = np.sin(2*np.pi*data.index.hour/24); data["hour_cos"] = np.cos(2*np.pi*data.index.hour/24)
data["dow_sin"]  = np.sin(2*np.pi*data.index.dayofweek/7); data["dow_cos"] = np.cos(2*np.pi*data.index.dayofweek/7)
data = data[["BDL_tmpf", "BDL_dwpf", "BDL_relh", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "Demand"]]   # Demand LAST
train, test = data.loc[:"2018-12-31"], data.loc["2019-01-01":]
print("train:", train.shape, "| test:", test.shape)
data.head()

in date order as delivered? False
train: (8760, 8) | test: (8760, 8)


,BDL_tmpf,BDL_dwpf,BDL_relh,hour_sin,hour_cos,dow_sin,dow_cos,Demand
Datetime,,,,,,,,
2018-01-01 00:00:00,1.9,0.0,56.85,0.000000,1.000000,0.0,1.0,3867.09
2018-01-01 01:00:00,0.0,0.0,62.12,0.258819,0.965926,0.0,1.0,3749.96
2018-01-01 02:00:00,0.0,0.0,61.38,0.500000,0.866025,0.0,1.0,3673.50
2018-01-01 03:00:00,0.0,0.0,71.11,0.707107,0.707107,0.0,1.0,3646.25
2018-01-01 04:00:00,0.0,0.0,64.21,0.866025,0.500000,0.0,1.0,3660.86


## (a) Two targets at once: demand *and* temperature

Move both targets to the right-hand side. `split_sequences_multi` takes the last `n_targets` columns at the next step as `y`; the inputs keep every column, so each target sees its own past.

In [3]:
cols2 = ["BDL_dwpf", "BDL_relh", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "BDL_tmpf", "Demand"]   # two targets LAST
tr2, te2 = train[cols2], test[cols2]
sc2   = MinMaxScaler().fit(tr2)
sc_t  = MinMaxScaler().fit(tr2[["BDL_tmpf", "Demand"]])   # target-only scaler to get real units back

def split_sequences_multi(seqs, n_steps, n_targets):
    X, y = [], []
    for i in range(len(seqs) - n_steps):
        X.append(seqs[i:i+n_steps, :]); y.append(seqs[i+n_steps, -n_targets:])
    return np.array(X), np.array(y)

n_steps = 24
X2_tr, y2_tr = split_sequences_multi(sc2.transform(tr2), n_steps, 2)
X2_te, y2_te = split_sequences_multi(sc2.transform(te2), n_steps, 2)
print("X:", X2_tr.shape, "| y:", y2_tr.shape, "-> two targets per sample")

X: (8736, 24, 8) | y: (8736, 2) -> two targets per sample


In [4]:
es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)

two = Sequential([LSTM(32, input_shape=(n_steps, X2_tr.shape[2])), Dense(2)])   # Dense(2, linear): one number per target
two.compile(optimizer="adam", loss="mse", metrics=["mae"])
two.summary()
two.fit(X2_tr, y2_tr, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)

pred2 = sc_t.inverse_transform(two.predict(X2_te, verbose=0)); true2 = sc_t.inverse_transform(y2_te)
print(f"temperature MAE: {mean_absolute_error(true2[:, 0], pred2[:, 0]):.2f} F   |   demand MAE: {mean_absolute_error(true2[:, 1], pred2[:, 1]):.1f} MW")

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         5,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,314 (20.76 KB)

 Trainable params: 5,314 (20.76 KB)

 Non-trainable params: 0 (0.00 B)

Restoring model weights from the end of the best epoch: 50.


temperature MAE: 1.41 F   |   demand MAE: 50.8 MW


In [5]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4)); i0 = 24*7*28; sl = slice(i0, i0+24*7)
ax[0].plot(true2[sl, 0], label="actual"); ax[0].plot(pred2[sl, 0], label="predicted"); ax[0].set_title("temperature (F), one test week"); ax[0].legend()
ax[1].plot(true2[sl, 1], label="actual"); ax[1].plot(pred2[sl, 1], label="predicted"); ax[1].set_title("demand (MW), same week"); ax[1].legend()
plt.tight_layout(); plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_8920\1781411880.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## (b) Multi-step: the next 24 hours of demand

One target, 24 steps. From the past **week**, predict tomorrow's 24 hourly values at once with `Dense(24)`.

In [6]:
sc   = MinMaxScaler().fit(train); sc_y = MinMaxScaler().fit(train[["Demand"]])
tr_s, te_s = sc.transform(train), sc.transform(test)

def split_multistep(seqs, n_in, n_out):
    X, y = [], []
    for i in range(len(seqs) - n_in - n_out + 1):
        X.append(seqs[i:i+n_in, :]); y.append(seqs[i+n_in:i+n_in+n_out, -1])
    return np.array(X), np.array(y)

n_in, n_out = 24*7, 24
Xs_tr, ys_tr = split_multistep(tr_s, n_in, n_out)
Xs_te, ys_te = split_multistep(te_s, n_in, n_out)
print("multi-step tensors:", Xs_tr.shape, "->", ys_tr.shape)

step = Sequential([LSTM(64, input_shape=(n_in, Xs_tr.shape[2])), Dense(n_out)])
step.compile(optimizer="adam", loss="mse", metrics=["mae"])
step.summary()
step.fit(Xs_tr, ys_tr, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=1)

multi-step tensors: (8569, 168, 8) -> (8569, 24)


C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,248 (79.09 KB)

...

106/108 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.0018 - mae: 0.0318

107/108 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.0018 - mae: 0.0317

108/108 ━━━━━━━━━━━━━━━━━━━━ 7s 61ms/step - loss: 0.0018 - mae: 0.0317 - val_loss: 0.0015 - val_mae: 0.0309


Epoch 41: early stopping


Restoring model weights from the end of the best epoch: 33.


In [7]:
pred_s = sc_y.inverse_transform(step.predict(Xs_te, verbose=0)); true_s = sc_y.inverse_transform(ys_te)
mae_by_h = np.abs(pred_s - true_s).mean(axis=0)

dem = test["Demand"].values     # seasonal-naive at every horizon: same hour yesterday
naive_by_h = np.array([mean_absolute_error(dem[n_in+h:len(dem)-n_out+h+1], dem[n_in+h-24:len(dem)-n_out+h+1-24]) for h in range(n_out)])

plt.figure(figsize=(8, 3.5))
plt.plot(range(1, 25), mae_by_h, marker="o", label="LSTM: past week -> next 24h")
plt.plot(range(1, 25), naive_by_h, marker="s", label="seasonal naive")
plt.xlabel("hours ahead"); plt.ylabel("test MAE (MW)"); plt.title("Error by horizon"); plt.legend(); plt.show()
print(f"model, averaged over 24 horizons: {mae_by_h.mean():.1f} MW   | seasonal naive: {naive_by_h.mean():.1f} MW")

model, averaged over 24 horizons: 176.2 MW   | seasonal naive: 226.9 MW


C:\Users\dww05002\AppData\Local\Temp\ipykernel_8920\3957505209.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("hours ahead"); plt.ylabel("test MAE (MW)"); plt.title("Error by horizon"); plt.legend(); plt.show()


In [8]:
d = 200
plt.figure(figsize=(8, 3)); plt.plot(true_s[d], marker="o", label="actual"); plt.plot(pred_s[d], marker="o", label="forecast made 24h earlier")
plt.xlabel("hour of the forecast day"); plt.ylabel("MW"); plt.title("A day-ahead forecast"); plt.legend(); plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_8920\1012564478.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("hour of the forecast day"); plt.ylabel("MW"); plt.title("A day-ahead forecast"); plt.legend(); plt.show()


## Save the model and use it again

In [9]:
step.save('E4_Many_To_Many_Demand_dayahead.keras')
reloaded = load_model('E4_Many_To_Many_Demand_dayahead.keras')
print("reloaded model reproduces the forecast:", np.allclose(step.predict(Xs_te[:5], verbose=0), reloaded.predict(Xs_te[:5], verbose=0)))

reloaded model reproduces the forecast: True


## On your own

- Add a **holiday flag** column - the model has never been told July 4th isn't a Tuesday.
- Swap `LSTM` for `GRU` in the day-ahead model. Fewer parameters; same error?
- Train on 2011-2019 and test on **2020**. That's what distribution shift does to a forecast.